In [1]:
import pandas as pd
import numpy as np 
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

In [2]:
# setup and feature engineering
df = pd.read_csv('../../data/processed/neighborhood_equity_model.csv')

# set neighborhood as index so removed from feature columns
df.set_index("neighborhood", inplace=True)

# select feature columns and scale
X = df.drop(columns=["avgOverallEquityScore"])

# target column - unscaled
target_col = 'avgOverallEquityScore'
y = df[target_col].round().astype(int).values

# only one instance with class 5 so merge with class 4:
y[y == 5] = 4

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# scale full X
scaler_full = StandardScaler()
X_scaled = scaler_full.fit_transform(X)

In [3]:
import numpy as np

classes, counts = np.unique(y, return_counts=True)

for cls, count in zip(classes, counts):
    print(f"Class {cls}: {count} instances")

Class 2: 28 instances
Class 3: 21 instances
Class 4: 29 instances


In [4]:
_, train_counts = np.unique(y_train, return_counts=True)
_, test_counts = np.unique(y_test, return_counts=True)
print("Train distribution:", dict(zip(classes, train_counts)))
print("Test distribution:", dict(zip(classes, test_counts)))

Train distribution: {2: 22, 3: 17, 4: 23}
Test distribution: {2: 6, 3: 4, 4: 6}


## Logist Regression Models

These models will use both the train/test split data, as well as the complete data set with cross_val_score to evaluate.

### Basic Logistic Regression without Regularization

In [5]:
# with trian_test_split
log_reg = LogisticRegression(max_iter=1000, penalty=None)

log_reg.fit(X_train_scaled, y_train)

# predict on the test set
y_pred = log_reg.predict(X_test_scaled)

# evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Confusion Matrix:
[[4 2 0]
 [1 3 0]
 [0 4 2]]

Classification Report:
              precision    recall  f1-score   support

           2       0.80      0.67      0.73         6
           3       0.33      0.75      0.46         4
           4       1.00      0.33      0.50         6

    accuracy                           0.56        16
   macro avg       0.71      0.58      0.56        16
weighted avg       0.76      0.56      0.58        16



In [6]:
# coefficients
coef_df = pd.DataFrame(
    log_reg.coef_,
    columns=X.columns,
    index=[f"Class {tier}" for tier in log_reg.classes_]
)
coef_df

,Percent Hispanic,Percent White,Percent Black,Percent Native American,Percent Asian,Percent Hawaiian/Pacific Islander,Percent Other Race,Percent Two or More Races,Median Household Income,Per Capita Income,Median Home Value,Percent Living in Poverty,population_density,percent_less_hs,percent_at_hs,percent_some_college,percent_bachelors,program_count_per1000,crime_per1000
Class 2,7.071755,-7.045656,4.262752,-0.363210,-13.820211,18.055444,9.583065,-4.046678,-9.964420,-2.296652,3.143089,10.602233,5.430135,2.785238,5.914873,-7.672859,-1.359173,-10.190467,1.665068
Class 3,-1.062803,0.064629,0.091005,1.721347,3.595646,-16.480431,0.787150,8.879386,1.569307,0.219416,-10.137990,4.396055,6.329766,3.164044,0.913539,-2.385068,-1.143418,7.027379,-0.860382
Class 4,-6.008952,6.981027,-4.353757,-1.358137,10.224565,-1.575013,-10.370215,-4.832708,8.395113,2.077236,6.994901,-14.998289,-11.759901,-5.949282,-6.828413,10.057928,2.502592,3.163088,-0.804685


In [7]:
# feature importance
feature_importance = coef_df.abs().sum(axis=0).sort_values(ascending=False)
print(feature_importance)

Percent Hawaiian/Pacific Islander    36.110888
Percent Living in Poverty            29.996577
Percent Asian                        27.640422
population_density                   23.519803
Percent Other Race                   20.740431
program_count_per1000                20.380934
Median Home Value                    20.275979
percent_some_college                 20.115855
Median Household Income              19.928840
Percent Two or More Races            17.758771
Percent Hispanic                     14.143509
Percent White                        14.091312
percent_at_hs                        13.656825
percent_less_hs                      11.898564
Percent Black                         8.707514
percent_bachelors                     5.005183
Per Capita Income                     4.593304
Percent Native American               3.442694
crime_per1000                         3.330135
dtype: float64


In [8]:
# with cross_val_score
log_reg_cv = LogisticRegression(max_iter=1000, penalty=None)
scores = cross_val_score(log_reg_cv, X_scaled, y, cv=5, scoring='f1_macro')
print("F1 scores:", scores)
print("Mean F1:", scores.mean())

F1 scores: [0.68080808 0.55218855 0.71904762 0.92207792 0.6031746 ]
Mean F1: 0.6954593554593556


### Logistic Regression with L1 Regularization

In [9]:
lasso = LogisticRegression(max_iter=10000, penalty='l1', C=0.5, solver='saga')

lasso.fit(X_train_scaled, y_train)

# predict on the test set
y_pred = lasso.predict(X_test_scaled)

# evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Confusion Matrix:
[[5 1 0]
 [1 2 1]
 [1 1 4]]

Classification Report:
              precision    recall  f1-score   support

           2       0.71      0.83      0.77         6
           3       0.50      0.50      0.50         4
           4       0.80      0.67      0.73         6

    accuracy                           0.69        16
   macro avg       0.67      0.67      0.67        16
weighted avg       0.69      0.69      0.69        16



In [10]:
# coefficients
coef_df = pd.DataFrame(
    lasso.coef_,  
    columns=X.columns,    
    index=[f"Class {tier}" for tier in lasso.classes_]
)
coef_df

,Percent Hispanic,Percent White,Percent Black,Percent Native American,Percent Asian,Percent Hawaiian/Pacific Islander,Percent Other Race,Percent Two or More Races,Median Household Income,Per Capita Income,Median Home Value,Percent Living in Poverty,population_density,percent_less_hs,percent_at_hs,percent_some_college,percent_bachelors,program_count_per1000,crime_per1000
Class 2,0.288706,-0.258888,0.0,0.0,-0.635984,0.649778,0.0,0.000000,0.000000,0.0,0.000000,0.857172,0.000000,0.0,0.244514,0.0,0.0,0.0,0.0
Class 3,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.366438,0.000000,0.0,0.000000,0.000000,0.057442,0.0,0.000000,0.0,0.0,0.0,0.0
Class 4,0.000000,0.748447,0.0,0.0,0.109585,0.000000,0.0,0.000000,0.390015,0.0,1.170139,-0.915701,-0.289553,0.0,0.000000,0.0,0.0,0.0,0.0


In [11]:
# feature importance
feature_importance = coef_df.abs().sum(axis=0).sort_values(ascending=False)
print(feature_importance)

Percent Living in Poverty            1.772873
Median Home Value                    1.170139
Percent White                        1.007335
Percent Asian                        0.745570
Percent Hawaiian/Pacific Islander    0.649778
Median Household Income              0.390015
Percent Two or More Races            0.366438
population_density                   0.346995
Percent Hispanic                     0.288706
percent_at_hs                        0.244514
program_count_per1000                0.000000
percent_bachelors                    0.000000
percent_some_college                 0.000000
Per Capita Income                    0.000000
percent_less_hs                      0.000000
Percent Other Race                   0.000000
Percent Native American              0.000000
Percent Black                        0.000000
crime_per1000                        0.000000
dtype: float64


In [12]:
# with cross_val_score
lasso_cv = LogisticRegression(max_iter=10000, penalty='l1', C=0.5, solver='saga')
scores = cross_val_score(lasso_cv, X_scaled, y, cv=5, scoring='f1_macro')
print("F1 scores:", scores)
print("Mean F1:", scores.mean())

F1 scores: [0.72466422 0.72610723 0.52777778 0.92207792 0.54047619]
Mean F1: 0.6882206682206682


### Logistic Regression with L2 Regularization

In [13]:
ridge = LogisticRegression(max_iter=1000, penalty='l2', C=0.5)

ridge.fit(X_train_scaled, y_train)

# predict on the test set
y_pred = ridge.predict(X_test_scaled)

# evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Confusion Matrix:
[[5 1 0]
 [1 2 1]
 [0 4 2]]

Classification Report:
              precision    recall  f1-score   support

           2       0.83      0.83      0.83         6
           3       0.29      0.50      0.36         4
           4       0.67      0.33      0.44         6

    accuracy                           0.56        16
   macro avg       0.60      0.56      0.55        16
weighted avg       0.63      0.56      0.57        16



In [14]:
# coefficients
coef_df = pd.DataFrame(
    ridge.coef_,  
    columns=X.columns,    
    index=[f"Class {tier}" for tier in ridge.classes_]
)
coef_df

,Percent Hispanic,Percent White,Percent Black,Percent Native American,Percent Asian,Percent Hawaiian/Pacific Islander,Percent Other Race,Percent Two or More Races,Median Household Income,Per Capita Income,Median Home Value,Percent Living in Poverty,population_density,percent_less_hs,percent_at_hs,percent_some_college,percent_bachelors,program_count_per1000,crime_per1000
Class 2,0.409737,-0.388738,0.145301,0.024890,-0.665106,0.622374,0.299777,-0.085792,-0.501932,-0.206718,0.089206,0.760652,0.219324,0.326937,0.303385,-0.155095,-0.234690,-0.230262,0.127280
Class 3,0.002121,-0.087255,0.105169,-0.027629,0.155583,-0.474414,0.057916,0.443085,0.054389,-0.062738,-0.554739,-0.020537,0.291590,0.149018,0.010514,-0.088822,-0.047787,-0.007597,-0.084871
Class 4,-0.411858,0.475993,-0.250470,0.002739,0.509523,-0.147960,-0.357693,-0.357293,0.447543,0.269456,0.465532,-0.740115,-0.510914,-0.475955,-0.313899,0.243917,0.282478,0.237859,-0.042409


In [15]:
# feature importance
feature_importance = coef_df.abs().sum(axis=0).sort_values(ascending=False)
print(feature_importance)

Percent Living in Poverty            1.521304
Percent Asian                        1.330211
Percent Hawaiian/Pacific Islander    1.244749
Median Home Value                    1.109477
population_density                   1.021827
Median Household Income              1.003864
Percent White                        0.951985
percent_less_hs                      0.951910
Percent Two or More Races            0.886170
Percent Hispanic                     0.823716
Percent Other Race                   0.715386
percent_at_hs                        0.627798
percent_bachelors                    0.564956
Per Capita Income                    0.538912
Percent Black                        0.500940
percent_some_college                 0.487833
program_count_per1000                0.475718
crime_per1000                        0.254560
Percent Native American              0.055258
dtype: float64


In [16]:
# with cross_val_score
ridge_cv = LogisticRegression(max_iter=1000, penalty='l2', C=0.5)
scores = cross_val_score(ridge_cv, X_scaled, y, cv=5, scoring='f1_macro')
print("F1 scores:", scores)
print("Mean F1:", scores.mean())

F1 scores: [0.62942613 0.58333333 0.57387057 1.         0.62433862]
Mean F1: 0.6821937321937322


### Logistic Regression with Elasticnet Regularization

In [17]:
elastic = LogisticRegression(max_iter=1000, penalty='elasticnet', l1_ratio=0.5, solver='saga')

elastic.fit(X_train_scaled, y_train)

# predict on the test set
y_pred = elastic.predict(X_test_scaled)

# evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Confusion Matrix:
[[5 1 0]
 [1 2 1]
 [0 4 2]]

Classification Report:
              precision    recall  f1-score   support

           2       0.83      0.83      0.83         6
           3       0.29      0.50      0.36         4
           4       0.67      0.33      0.44         6

    accuracy                           0.56        16
   macro avg       0.60      0.56      0.55        16
weighted avg       0.63      0.56      0.57        16



In [18]:
# coefficients
coef_df = pd.DataFrame(
    elastic.coef_,  
    columns=X.columns,    
    index=[f"Class {tier}" for tier in elastic.classes_]
)
coef_df

,Percent Hispanic,Percent White,Percent Black,Percent Native American,Percent Asian,Percent Hawaiian/Pacific Islander,Percent Other Race,Percent Two or More Races,Median Household Income,Per Capita Income,Median Home Value,Percent Living in Poverty,population_density,percent_less_hs,percent_at_hs,percent_some_college,percent_bachelors,program_count_per1000,crime_per1000
Class 2,0.412805,-0.547848,0.00000,0.0,-0.891781,0.969840,0.209582,0.000000,-0.804650,0.000000,0.000000,1.037756,0.000000,0.084894,0.346778,0.000000,0.000000,-0.142880,0.0
Class 3,0.000000,0.000000,0.00000,0.0,0.000000,-0.436127,0.000000,0.573828,0.000000,0.000000,-0.767940,0.000000,0.026599,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
Class 4,-0.465916,0.811552,-0.17195,0.0,0.524299,0.000000,-0.398027,-0.202927,0.454602,0.045739,0.623375,-1.193409,-0.785263,-0.656789,-0.222530,0.008162,0.004289,0.091264,0.0


In [19]:
# feature importance
feature_importance = coef_df.abs().sum(axis=0).sort_values(ascending=False)
print(feature_importance)

Percent Living in Poverty            2.231165
Percent Asian                        1.416080
Percent Hawaiian/Pacific Islander    1.405967
Median Home Value                    1.391315
Percent White                        1.359401
Median Household Income              1.259251
Percent Hispanic                     0.878721
population_density                   0.811862
Percent Two or More Races            0.776755
percent_less_hs                      0.741683
Percent Other Race                   0.607609
percent_at_hs                        0.569308
program_count_per1000                0.234144
Percent Black                        0.171950
Per Capita Income                    0.045739
percent_some_college                 0.008162
percent_bachelors                    0.004289
Percent Native American              0.000000
crime_per1000                        0.000000
dtype: float64


In [20]:
# with cross_val_score
elastic_cv = LogisticRegression(max_iter=1000, penalty='elasticnet', l1_ratio=0.5, solver='saga')
scores = cross_val_score(elastic_cv, X_scaled, y, cv=5, scoring='f1_macro')
print("F1 scores:", scores)
print("Mean F1:", scores.mean())

F1 scores: [0.72610723 0.58333333 0.57387057 1.         0.6981685 ]
Mean F1: 0.7162959262959263


## Hyperparameter tuning

Using GridSearchCV to find regularization hyperparameters

In [21]:
# setup
model_cv = LogisticRegression(solver='saga', max_iter=10000)

# setting up grid of tunable hyperparameters
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'C': [0.01, 0.1, 0.5, 1, 10],
    'l1_ratio': [0, 0.25, 0.5, 0.75, 1]
}

grid_search = GridSearchCV(
    estimator=model_cv,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    refit=True,
    n_jobs=-1
)

# fit on entire scaled feature columns
grid_search.fit(X_scaled, y)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warni

Best params: {'C': 10, 'l1_ratio': 0.5, 'penalty': 'elasticnet'}
Best score: 0.7539675139675139


/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


In [22]:
# model using the hyperparameters found above
elastic_update = LogisticRegression(max_iter=10000, penalty='elasticnet', C=10, l1_ratio=0.5, solver='saga')

elastic_update.fit(X_train_scaled, y_train)

# predict on the test set
y_pred = elastic_update.predict(X_test_scaled)

# evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Confusion Matrix:
[[5 1 0]
 [1 3 0]
 [0 4 2]]

Classification Report:
              precision    recall  f1-score   support

           2       0.83      0.83      0.83         6
           3       0.38      0.75      0.50         4
           4       1.00      0.33      0.50         6

    accuracy                           0.62        16
   macro avg       0.74      0.64      0.61        16
weighted avg       0.78      0.62      0.62        16



In [23]:
# coefficients
coef_df = pd.DataFrame(
    elastic_update.coef_,  
    columns=X.columns,    
    index=[f"Class {tier}" for tier in elastic_update.classes_]
)
coef_df

,Percent Hispanic,Percent White,Percent Black,Percent Native American,Percent Asian,Percent Hawaiian/Pacific Islander,Percent Other Race,Percent Two or More Races,Median Household Income,Per Capita Income,Median Home Value,Percent Living in Poverty,population_density,percent_less_hs,percent_at_hs,percent_some_college,percent_bachelors,program_count_per1000,crime_per1000
Class 2,1.694987,-1.592309,0.208023,0.028413,-2.670283,3.420322,1.578708,-0.788071,-1.840881,-0.066405,0.157729,2.166711,0.661286,0.029113,0.974978,-0.490714,0.000000,-1.729586,0.023993
Class 3,0.000000,0.000000,0.000000,-0.064297,0.429190,-2.502564,0.000000,1.586836,0.191080,0.000000,-2.022359,0.123478,0.579987,0.664566,0.000000,0.000000,0.000000,0.483506,0.000000
Class 4,-1.540949,1.814794,-0.550720,0.000000,1.535743,-0.212170,-1.607620,-0.093168,0.956591,0.561326,1.164690,-2.972352,-1.946414,-1.396307,-0.949970,1.182515,0.395837,0.554813,-0.003812


In [24]:
# feature importance
feature_importance = coef_df.abs().sum(axis=0).sort_values(ascending=False)
print(feature_importance)

Percent Hawaiian/Pacific Islander    6.135056
Percent Living in Poverty            5.262540
Percent Asian                        4.635215
Percent White                        3.407102
Median Home Value                    3.344779
Percent Hispanic                     3.235935
population_density                   3.187688
Percent Other Race                   3.186328
Median Household Income              2.988553
program_count_per1000                2.767906
Percent Two or More Races            2.468075
percent_less_hs                      2.089986
percent_at_hs                        1.924948
percent_some_college                 1.673229
Percent Black                        0.758744
Per Capita Income                    0.627731
percent_bachelors                    0.395837
Percent Native American              0.092710
crime_per1000                        0.027805
dtype: float64


## Model Updates

Updating the feature columns based on the regularization models above

In [25]:
# new feature columns
new_features = ["Percent Hawaiian/Pacific Islander", "Percent Living in Poverty", "Median Household Income", "percent_less_hs", "Median Home Value", "Percent Hispanic", "Percent Asian", "Percent White"]

X = df[new_features]

# target column - unscaled
target_col = 'avgOverallEquityScore'
y = df[target_col].round().astype(int).values

# only one instance with class 5 so merge with class 4:
y[y == 5] = 4

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# scale full X
scaler_full = StandardScaler()
X_scaled = scaler_full.fit_transform(X)

### Using GridSearchCV with new features

In [26]:
# setup
new_model_cv = LogisticRegression(solver='saga', max_iter=10000)

# setting up grid of tunable hyperparameters
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'C': [0.01, 0.1, 0.5, 1, 10],
    'l1_ratio': [0, 0.25, 0.5, 0.75, 1]
}

grid_search = GridSearchCV(
    estimator=new_model_cv,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    refit=True,
    n_jobs=-1
)

# fit on entire scaled feature columns
grid_search.fit(X_scaled, y)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warni

Best params: {'C': 10, 'l1_ratio': 0, 'penalty': 'elasticnet'}
Best score: 0.8102430902430902


/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/uelski/anaconda3/lib/python3.11/site-packages/sklearn

In [27]:
# final model with the hyperparameters above
new_elastic_update = LogisticRegression(max_iter=10000, penalty='elasticnet', C=10, l1_ratio=0, solver='saga')

new_elastic_update.fit(X_train_scaled, y_train)

# predict on the test set
y_pred = new_elastic_update.predict(X_test_scaled)

# evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Confusion Matrix:
[[5 1 0]
 [1 1 2]
 [0 2 4]]

Classification Report:
              precision    recall  f1-score   support

           2       0.83      0.83      0.83         6
           3       0.25      0.25      0.25         4
           4       0.67      0.67      0.67         6

    accuracy                           0.62        16
   macro avg       0.58      0.58      0.58        16
weighted avg       0.62      0.62      0.62        16



In [28]:
# coefficients
coef_df = pd.DataFrame(
    new_elastic_update.coef_,  
    columns=X.columns,    
    index=[f"Class {tier}" for tier in new_elastic_update.classes_]
)
coef_df

,Percent Hawaiian/Pacific Islander,Percent Living in Poverty,Median Household Income,percent_less_hs,Median Home Value,Percent Hispanic,Percent Asian,Percent White
Class 2,2.471365,2.744128,-2.480133,-0.334093,0.094860,1.763617,-1.242670,-1.576642
Class 3,-1.824561,0.745601,0.572424,1.471480,-2.247295,-0.843175,-0.230713,0.271180
Class 4,-0.646804,-3.489729,1.907708,-1.137387,2.152435,-0.920442,1.473383,1.305461


In [29]:
# feature importance
feature_importance = coef_df.abs().sum(axis=0).sort_values(ascending=False)
print(feature_importance)

Percent Living in Poverty            6.979459
Median Household Income              4.960265
Percent Hawaiian/Pacific Islander    4.942730
Median Home Value                    4.494590
Percent Hispanic                     3.527234
Percent White                        3.153283
Percent Asian                        2.946766
percent_less_hs                      2.942960
dtype: float64
